# Individual Model Analysis — Concordant Good/Poor Case Selection

Loads Dice scores from all three segmentation models (3D U-Net, nnUNet, TransBTS),
identifies cases where all three models fail (concordant-poor) or succeed (concordant-good)
simultaneously, then combines radiomic feature matrices for downstream oracle training.

**Inputs:**
- `../Results/Result/` — per-model Dice CSVs / JSON summaries
- `../Results/Analysis_Results/` — per-feature-family pickle files

**Outputs:**
- `results_df` — combined feature + Dice matrix for all cases
- `Feature_list` — dict mapping feature family \u2192 column names

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import copy
import pickle as pkl

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import mutual_info_regression

from hyperopt import hp, fmin, tpe, Trials, STATUS_OK

## Data Loading Utilities

In [ ]:
def load_unet_result(path, verbose=False):
    """Load 3D U-Net Dice CSV; strip the '-seg' suffix from case IDs."""
    df = pd.read_csv(path, index_col='Unnamed: 0')
    df.index = [idx.split('-seg')[0] for idx in df.index]
    df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard'], axis=1, inplace=True)
    summary = pd.DataFrame({'mean': df.mean(), 'std': df.std()})
    if verbose:
        print('==== 3D U-Net ===='); print(summary)
    return summary, df


def load_nnunet_result(path, verbose=False):
    """Load nnUNet summary.json; extract per-case WT/TC/ET Dice."""
    with open(path) as f:
        data = json.load(f)

    rows = []
    for case in data['metric_per_case']:
        rows.append({
            'case':    case['reference_file'].split('/')[-1].split('.')[0],
            # nnUNet uses label-set tuples as metric keys: (2,1,3)=WT, (2,3)=TC, (3,)=ET
            'WT dice': case['metrics']['(2, 1, 3)']['Dice'],
            'TC dice': case['metrics']['(2, 3)']['Dice'],
            'ET dice': case['metrics']['(3,)']['Dice'],
        })
    df = pd.DataFrame(rows).set_index('case')
    summary = pd.DataFrame({'mean': df.mean(), 'std': df.std()})
    if verbose:
        print('==== nnUNet ===='); print(summary)
    return summary, df


def load_transbts_result(path, verbose=False):
    """Load TransBTS summary JSON; extract per-case WT/TC/ET Dice."""
    with open(path) as f:
        data = json.load(f)

    rows = [{
        'case':    cid,
        'WT dice': v['WT'][0],
        'TC dice': v['TC'][0],
        'ET dice': v['ET'][0],
    } for cid, v in data.items()]
    df = pd.DataFrame(rows).set_index('case')
    summary = pd.DataFrame({'mean': df.mean(), 'std': df.std()})
    if verbose:
        print('==== TransBTS ===='); print(summary)
    return summary, df


def read_results(verbose=True):
    """Load all three model result files and return raw per-case DataFrames."""
    _, unet_df = load_unet_result(
        '../Results/Result/Vanilla_Unet/Unet_test_dice.csv', verbose)
    _, nnunet_da_df = load_nnunet_result(
        '../Results/Result/nnUnet/nnUNetTrainer/summary.json', verbose)
    _, nnunet_noda_df = load_nnunet_result(
        '../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json', verbose)
    _, transbts_df = load_transbts_result(
        '../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json', verbose)
    return unet_df, nnunet_noda_df, nnunet_da_df, transbts_df

## Concordant Case Selection

A **concordant-poor** case is one where all three models fall below the Dice thresholds
simultaneously. Thresholds are the 25th-percentile Dice scores reported in the paper.

In [ ]:
def get_overlaps(unet_df, transbts_df, nnunet_noda_df,
                 wt_thresh, tc_thresh, et_thresh):
    """Return cases where all three models are simultaneously below the Dice thresholds."""
    def below(df):
        return set(df[
            (df['WT dice'] < wt_thresh) &
            (df['TC dice'] < tc_thresh) &
            (df['ET dice'] < et_thresh)
        ].index)

    all_poor      = below(unet_df) & below(transbts_df) & below(nnunet_noda_df)
    unet_nnunet   = below(unet_df) & below(nnunet_noda_df)
    return list(all_poor), list(unet_nnunet)

In [ ]:
# Thresholds from the paper (25th-percentile Dice across models)
WT_THRESH = 0.91
TC_THRESH = 0.86
ET_THRESH = 0.85

unet_df, nnunet_noda_df, nnunet_da_df, transbts_df = read_results(verbose=False)
performance_df = unet_df  # U-Net scores used as the oracle prediction target

all_poor_cases, unet_nnunet_poor = get_overlaps(
    unet_df, transbts_df, nnunet_noda_df,
    WT_THRESH, TC_THRESH, ET_THRESH,
)
print(f'Concordant-poor cases (all 3 models): {len(all_poor_cases)}')
print(f'Concordant-poor cases (U-Net + nnUNet): {len(unet_nnunet_poor)}')

## Feature Loading

Each feature family is stored as a pickle dict in `../Results/Analysis_Results/<family>/`.
The helper below loads one family, merges it with the Dice performance DataFrame,
and returns the joined table.

In [ ]:
LOCATION = 'Tumor_WT'


def load_radiomics_pkl(analysis_type: str) -> dict:
    path = f'../Results/Analysis_Results/Radiomics/{LOCATION}/{analysis_type}.pkl'
    with open(path, 'rb') as f:
        return pkl.load(f)


def get_feature_df(performance_df: pd.DataFrame, analysis_types: list) -> pd.DataFrame:
    """Merge radiomic features from one or more analysis types with the performance table."""
    results_df = performance_df.copy()
    for atype in analysis_types:
        try:
            raw = load_radiomics_pkl(atype)
        except FileNotFoundError as e:
            print(f'Skip {atype}: {e}')
            continue
        for prop_name, prop_data in raw.items():
            prop_df = pd.DataFrame.from_dict(prop_data, orient='index').astype(float)
            prop_df.columns = [f'{prop_name}_{c}_{atype}' for c in prop_df.columns]
            results_df = results_df.merge(prop_df, left_index=True, right_index=True)

    results_df.drop(['WT dice', 'TC dice', 'ET dice'], axis=1, inplace=True)
    return results_df

### Load Each Feature Family

In [ ]:
SHAPE_FEATURES = [
    'original_shape_Elongation_flair_shape',
    'original_shape_Flatness_flair_shape',
    'original_shape_LeastAxisLength_flair_shape',
    'original_shape_MajorAxisLength_flair_shape',
    'original_shape_MinorAxisLength_flair_shape',
    'original_shape_Sphericity_flair_shape',
]
shape_df = get_feature_df(performance_df, ['shape'])[SHAPE_FEATURES]

SIZE_FEATURES = [
    'diagnostics_Mask-original_VolumeNum_flair_size',
    'original_shape_MeshVolume_flair_size',
    'original_shape_SurfaceArea_flair_size',
    'original_shape_SurfaceVolumeRatio_flair_size',
]
size_df = get_feature_df(performance_df, ['size'])[SIZE_FEATURES]

INTENSITY_FEATURES = [
    'diagnostics_Image-original_Mean_flair_intensity',
    'diagnostics_Image-original_Mean_t2_intensity',
    'diagnostics_Image-original_Mean_t1_intensity',
    'diagnostics_Image-original_Mean_t1ce_intensity',
    'diagnostics_Image-original_Maximum_flair_intensity',
    'diagnostics_Image-original_Maximum_t2_intensity',
    'diagnostics_Image-original_Maximum_t1_intensity',
    'diagnostics_Image-original_Maximum_t1ce_intensity',
]
intensity_df = get_feature_df(performance_df, ['intensity'])[INTENSITY_FEATURES]

# First-order stats: 5 statistics x 4 modalities
FIRSTORDER_FEATURES = [
    f'original_firstorder_{stat}_{mod}_firstorder'
    for stat in ['Energy', 'Entropy', 'Kurtosis', 'Skewness', 'Uniformity']
    for mod  in ['flair', 't2', 't1', 't1ce']
]
firstorder_df = get_feature_df(performance_df, ['firstorder'])[FIRSTORDER_FEATURES]

print('shape:', shape_df.shape, '| size:', size_df.shape,
      '| intensity:', intensity_df.shape, '| firstorder:', firstorder_df.shape)

In [ ]:
# Texture features: each matrix type x selected statistics x 4 modalities
NGTDM_FEATURES = [
    f'original_ngtdm_{feat}_{mod}_ngtdm_10'
    for feat in ['Busyness', 'Coarseness', 'Complexity', 'Contrast', 'Strength']
    for mod  in ['flair', 't2', 't1', 't1ce']
]
ngtdm_df = get_feature_df(performance_df, ['ngtdm_10'])[NGTDM_FEATURES]

GLCM_FEATURES = [
    f'original_glcm_{feat}_{mod}_glcm_10'
    for feat in ['Autocorrelation', 'ClusterProminence', 'ClusterShade', 'ClusterTendency',
                 'Contrast', 'Correlation', 'JointAverage', 'JointEnergy', 'JointEntropy', 'MCC']
    for mod  in ['flair', 't2', 't1', 't1ce']
]
glcm_df = get_feature_df(performance_df, ['glcm_10'])[GLCM_FEATURES]

GLDM_FEATURES = [
    f'original_gldm_{feat}_{mod}_gldm_10'
    for feat in ['DependenceNonUniformity', 'GrayLevelNonUniformity', 'GrayLevelVariance',
                 'HighGrayLevelEmphasis', 'LargeDependenceEmphasis',
                 'LowGrayLevelEmphasis', 'SmallDependenceEmphasis']
    for mod  in ['flair', 't2', 't1', 't1ce']
]
gldm_df = get_feature_df(performance_df, ['gldm_10'])[GLDM_FEATURES]

GLRLM_FEATURES = [
    f'original_glrlm_{feat}_{mod}_glrlm'
    for feat in ['GrayLevelNonUniformity', 'LongRunEmphasis', 'LongRunHighGrayLevelEmphasis',
                 'LongRunLowGrayLevelEmphasis', 'LowGrayLevelRunEmphasis',
                 'RunLengthNonUniformity', 'RunPercentage', 'ShortRunEmphasis',
                 'ShortRunHighGrayLevelEmphasis', 'ShortRunLowGrayLevelEmphasis']
    for mod  in ['flair', 't2', 't1', 't1ce']
]
glrlm_df = get_feature_df(performance_df, ['glrlm'])[GLRLM_FEATURES]

GLSZM_FEATURES = [
    f'original_glszm_{feat}_{mod}_glszm'
    for feat in ['LargeAreaEmphasis', 'SizeZoneNonUniformity', 'SmallAreaEmphasis', 'ZoneEntropy']
    for mod  in ['flair', 't2', 't1', 't1ce']
]
glszm_df = get_feature_df(performance_df, ['glszm'])[GLSZM_FEATURES]

print('ngtdm:', ngtdm_df.shape, '| glcm:', glcm_df.shape, '| gldm:', gldm_df.shape,
      '| glrlm:', glrlm_df.shape, '| glszm:', glszm_df.shape)

In [ ]:
volume_df = pd.read_csv('../Results/Analysis_Results/volume/GLI-Tumor_volumns.csv',
                        index_col='Unnamed: 0')
volume_df = volume_df[['ED', 'ET', 'NCR', 'WT_volume', 'TC_volume', 'ET_volume',
                        'TC_WT_ratio', 'ET_WT_ratio', 'ET_TC_ratio']]

curvature_df = pd.read_csv('../Results/Analysis_Results/curverature/curverature.csv',
                           index_col='Unnamed: 0')
curvature_df = curvature_df[['mean_gaussian_curvature', 'std_gaussian_curvature',
                              'pos', 'neg', 'pos_count', 'neg_count']]

saliency_df = pd.read_csv('../Results/Analysis_Results/Saliency/Saliency.csv',
                          index_col='Unnamed: 0')

probability_df = pd.read_csv('../Results/Analysis_Results/probability/Probability_Tumor_boundary.csv',
                             index_col='Unnamed: 0')

print('volume:', volume_df.shape, '| curvature:', curvature_df.shape,
      '| saliency:', saliency_df.shape, '| probability:', probability_df.shape)

## Combine All Features

In [ ]:
# Registry of feature families \u2192 column lists (used for per-family oracle training)
Feature_list = {
    'shape':       SHAPE_FEATURES,
    'size':        SIZE_FEATURES,
    'intensity':   INTENSITY_FEATURES,
    'volume':      volume_df.columns.tolist(),
    'curvature':   curvature_df.columns.tolist(),
    'saliency':    saliency_df.columns.tolist(),
    'probability': probability_df.columns.tolist(),
    'firstorder':  FIRSTORDER_FEATURES,
    'ngtdm':       NGTDM_FEATURES,
    'glcm':        GLCM_FEATURES,
    'gldm':        GLDM_FEATURES,
    'glrlm':       GLRLM_FEATURES,
    'glszm':       GLSZM_FEATURES,
}

# Merge all feature families, then append Dice scores as targets
feature_frames = [
    shape_df, size_df, intensity_df, volume_df,
    curvature_df, saliency_df, probability_df,
    firstorder_df, ngtdm_df, glcm_df, gldm_df, glrlm_df, glszm_df,
]
results_df = feature_frames[0].copy()
for fdf in feature_frames[1:]:
    results_df = results_df.merge(fdf, left_index=True, right_index=True)
results_df = results_df.merge(performance_df, left_index=True, right_index=True)

results_df = results_df.dropna()
print(f'Combined feature matrix: {results_df.shape}')

## Oracle Model — Per-Family Gradient Boosting

Train one Gradient Boosting Regressor per feature family to predict WT Dice,
then combine via weighted ensemble (weights \u221d 1/validation-MAE).

In [ ]:
def train_gbr(X_train, y_train):
    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                      max_depth=2, random_state=42)
    model.fit(X_train, y_train)
    return model


def normalize_data(X: pd.DataFrame):
    scaler   = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    return scaler, X_scaled

In [ ]:
results_df.reset_index(drop=True, inplace=True)
X = results_df.drop(['WT dice', 'TC dice', 'ET dice'], axis=1)
y = results_df['WT dice']

scaler, X = normalize_data(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# hold out a small validation slice to compute per-family ensemble weights
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.2, random_state=42)
print(f'Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}')

In [ ]:
model_list = []
for family, cols in Feature_list.items():
    m = train_gbr(X_train[cols], y_train)
    model_list.append(m)
    print(f'{family:12s}  train MAE: {mean_absolute_error(y_train, m.predict(X_train[cols])):.4f}')

In [ ]:
families = list(Feature_list.keys())

# Compute per-family validation MAE \u2192 invert for ensemble weights
val_maes = [
    mean_absolute_error(y_val, model_list[i].predict(X_val[Feature_list[families[i]]]))
    for i in range(len(families))
]
weights = 1.0 / np.array(val_maes)
weights /= weights.sum()

# Weighted ensemble prediction on the held-out test set
test_preds = np.stack([
    model_list[i].predict(X_test[Feature_list[families[i]]])
    for i in range(len(families))
])
weighted_pred = np.average(test_preds, axis=0, weights=weights)

print(f'Weighted ensemble \u2014 Test MAE: {mean_absolute_error(y_test, weighted_pred):.4f}')
print(f'                    Test R\u00b2:  {r2_score(y_test, weighted_pred):.4f}')

## Median Ensemble

In [ ]:
preds_df    = pd.DataFrame(test_preds, index=families)
median_pred = preds_df.median()

mse = mean_squared_error(y_test, median_pred)
mae = mean_absolute_error(y_test, median_pred)
r2  = r2_score(y_test, median_pred)
print(f'Median ensemble \u2014 MSE: {mse:.4f}  MAE: {mae:.4f}  R\u00b2: {r2:.4f}')